# ⚙️ Optimizers & Normalization — From Scratch

> **Month 1 · Week 2 Deliverable** | ML Safety Program  
> Author: Inés | [GitHub](https://github.com/)

---

## Overview

This self-contained notebook implements and compares key **optimization algorithms** and **normalization techniques** from scratch using only NumPy and pure Python — no PyTorch autograd for the core implementations.

The motivation: understanding *why* these components work (and when they fail) is essential for diagnosing training instabilities in large models — a critical skill in ML Safety.

### Contents
| Section | Topic |
|---------|-------|
| 1 | SGD, Momentum, AdaGrad, Adam — scalar implementations |
| 2 | Comparative convergence experiment on a 2D ill-conditioned loss |
| 3 | Adam failure mode: non-convergence on adversarial gradient sequences |
| 4 | BatchNorm vs LayerNorm — from-scratch with full backward pass |
| 5 | Normalization comparison in a Transformer-like architecture |
| 6 | Learning rate schedules: cosine annealing + warmup |
| 7 | Conclusions: when to use each technique |

---


## 0. Imports

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from dataclasses import dataclass, field
from typing import List

# Reproducibility
np.random.seed(42)
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.spines.top': False,
    'axes.spines.right': False,
})


## 1. Optimizer Implementations

All optimizers operate on a minimal `Param` container that mimics PyTorch's parameter abstraction.

### Design principle
Each optimizer implements two methods:
- `step()` — apply one gradient update using the current `p.grad`
- `zero_grad()` — reset all gradients to zero before the next forward pass

This mirrors the PyTorch API exactly.


In [ ]:
class Param:
    """Minimal parameter container (mimics torch.nn.Parameter)."""
    def __init__(self, value: float):
        self.data = value
        self.grad = 0.0

    def __repr__(self):
        return f"Param(data={self.data:.4f}, grad={self.grad:.4f})"


### 1.1 SGD

The vanilla update rule:

$$\theta_{t+1} = \theta_t - \alpha \nabla_{\theta} \mathcal{L}$$

Simple, stable, but treats all parameters identically regardless of their gradient history or scale.


In [ ]:
class SGD:
    """Stochastic Gradient Descent."""
    def __init__(self, params, lr=0.01):
        self.params = list(params)
        self.lr     = lr

    def step(self):
        for p in self.params:
            if p.grad is None:
                continue
            p.data -= self.lr * p.grad

    def zero_grad(self):
        for p in self.params:
            p.grad = 0.0


### 1.2 SGD + Momentum

Momentum maintains a running average of past gradients (the "velocity" $v$):

$$v_t = \beta v_{t-1} + (1-\beta)\, g_t$$
$$\theta_{t+1} = \theta_t - \alpha v_t$$

**Why it works:** in directions where the gradient is consistently positive (or negative), the velocity builds up and the update accelerates. In oscillatory directions, the opposing gradients cancel in the velocity, dampening oscillations. This is crucial for ill-conditioned loss surfaces (very different curvature in different directions).


In [ ]:
class Momentum:
    """SGD with exponential moving average of gradients."""
    def __init__(self, params, lr=0.01, beta=0.9):
        self.params = list(params)
        self.lr     = lr
        self.beta   = beta
        self.v      = [0.0] * len(self.params)

    def step(self):
        for i, p in enumerate(self.params):
            if p.grad is None:
                continue
            self.v[i] = self.beta * self.v[i] + (1 - self.beta) * p.grad
            p.data   -= self.lr * self.v[i]

    def zero_grad(self):
        for p in self.params:
            p.grad = 0.0


### 1.3 AdaGrad

Accumulates squared gradients and uses them to adapt the learning rate per parameter:

$$r_t = r_{t-1} + g_t^2$$
$$\theta_{t+1} = \theta_t - \frac{\alpha}{\sqrt{r_t} + \varepsilon}\, g_t$$

**Key insight:** parameters that receive large gradients frequently get their effective LR reduced, while rarely-updated parameters maintain a larger effective LR. This is very useful for sparse features (NLP).

**Weakness:** $r_t$ grows monotonically — the effective LR decreases forever and eventually reaches effectively zero.


In [ ]:
class AdaGrad:
    """Adaptive per-parameter learning rates via accumulated squared gradients."""
    def __init__(self, params, lr=0.01, eps=1e-8):
        self.params = list(params)
        self.lr     = lr
        self.eps    = eps
        self.r      = [0.0] * len(self.params)

    def step(self):
        for i, p in enumerate(self.params):
            if p.grad is None:
                continue
            self.r[i] += p.grad ** 2
            p.data    -= (self.lr / (math.sqrt(self.r[i]) + self.eps)) * p.grad

    def zero_grad(self):
        for p in self.params:
            p.grad = 0.0


### 1.4 Adam

Combines momentum (first moment $m$) with AdaGrad-style adaptive rates (second moment $v$):

$$m_t = \beta_1 m_{t-1} + (1-\beta_1)\, g_t$$
$$v_t = \beta_2 v_{t-1} + (1-\beta_2)\, g_t^2$$
$$\hat{m}_t = \frac{m_t}{1 - \beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1 - \beta_2^t}$$
$$\theta_{t+1} = \theta_t - \frac{\alpha \hat{m}_t}{\sqrt{\hat{v}_t} + \varepsilon}$$

**Bias correction** ($1 - \beta^t$): at step $t=1$, without correction, $m_1 = (1-\beta_1)g_1 \approx 0.1 g_1$ — far too small. Correction removes this initialisation bias, making Adam stable from step 1.


In [ ]:
class Adam:
    """
    Adam: Adaptive Moment Estimation.
    Default hyperparams from the original paper (Kingma & Ba, 2015).
    """
    def __init__(self, params, lr=0.001, beta1=0.9, beta2=0.999, eps=1e-8):
        self.params = list(params)
        self.lr     = lr
        self.beta1  = beta1
        self.beta2  = beta2
        self.eps    = eps
        self.m      = [0.0] * len(self.params)
        self.v      = [0.0] * len(self.params)
        self.t      = 0

    def step(self):
        self.t += 1
        for i, p in enumerate(self.params):
            if p.grad is None:
                continue
            g          = p.grad
            self.m[i]  = self.beta1 * self.m[i] + (1 - self.beta1) * g
            self.v[i]  = self.beta2 * self.v[i] + (1 - self.beta2) * g ** 2
            m_hat      = self.m[i] / (1 - self.beta1 ** self.t)
            v_hat      = self.v[i] / (1 - self.beta2 ** self.t)
            p.data    -= self.lr * m_hat / (math.sqrt(v_hat) + self.eps)

    def zero_grad(self):
        for p in self.params:
            p.grad = 0.0


## 2. Comparative Convergence Experiment

### Test function: ill-conditioned quadratic

$$f(x, y) = x^2 + 100 y^2, \quad \nabla f = (2x,\, 200y)$$

The Hessian has eigenvalues $\{2, 200\}$ — a **condition number of 100**. This means the loss surface is a very elongated ellipse, the classic stress-test for optimizers: vanilla SGD oscillates wildly along the steep $y$ dimension while making slow progress along $x$.

Starting point: $(5.0, 5.0)$.


In [ ]:
def f(x, y):
    return x**2 + 100 * y**2

def grad_f(x, y):
    return 2*x, 200*y   # analytical gradient


def run_optimizer(optimizer_factory, steps=200):
    x = Param(5.0)
    y = Param(5.0)
    opt = optimizer_factory([x, y])

    traj_x, traj_y, losses = [], [], []
    for _ in range(steps):
        dx, dy = grad_f(x.data, y.data)
        x.grad, y.grad = dx, dy

        traj_x.append(x.data)
        traj_y.append(y.data)
        losses.append(f(x.data, y.data))

        opt.step()
        opt.zero_grad()

    return traj_x, traj_y, losses


STEPS = 200

results = {
    "SGD":      run_optimizer(lambda p: SGD(p, lr=0.005),              STEPS),
    "Momentum": run_optimizer(lambda p: Momentum(p, lr=0.005, beta=0.9), STEPS),
    "AdaGrad":  run_optimizer(lambda p: AdaGrad(p, lr=0.5),            STEPS),
    "Adam":     run_optimizer(lambda p: Adam(p, lr=0.2),               STEPS),
}

COLORS = {"SGD": "steelblue", "Momentum": "darkorange",
          "AdaGrad": "mediumseagreen", "Adam": "tomato"}


In [ ]:
fig = plt.figure(figsize=(16, 6))
gs  = gridspec.GridSpec(1, 2, wspace=0.35)

# ── Left: trajectories ──────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0])
x_grid = np.linspace(-6, 6, 300)
y_grid = np.linspace(-6, 6, 300)
X, Y   = np.meshgrid(x_grid, y_grid)
Z      = X**2 + 100 * Y**2

levels = np.logspace(0, 4, 20)
ax1.contour(X, Y, Z, levels=levels, cmap='Greys', linewidths=0.8, alpha=0.6)
ax1.contourf(X, Y, Z, levels=levels, cmap='Greys', alpha=0.1)

for name, (tx, ty, _) in results.items():
    ax1.plot(tx, ty, '-o', label=name, color=COLORS[name], lw=1.5,
             markersize=2, alpha=0.85)

ax1.scatter([5.0], [5.0], s=120, zorder=10, color='black', marker='*', label='Start')
ax1.scatter([0.0], [0.0], s=120, zorder=10, color='gold',  marker='*', label='Optimum')
ax1.set_xlim(-6, 6); ax1.set_ylim(-2, 6)
ax1.set_xlabel("x"); ax1.set_ylabel("y")
ax1.set_title("Optimization Trajectories\n$f(x,y) = x^2 + 100y^2$",
              fontweight='bold')
ax1.legend(loc='upper right', fontsize=9)

# ── Right: loss curves (log scale) ─────────────────────────────────────────
ax2 = fig.add_subplot(gs[1])
for name, (_, _, losses) in results.items():
    ax2.semilogy(losses, label=name, color=COLORS[name], lw=2)

ax2.set_xlabel("Iteration")
ax2.set_ylabel("f(x, y)  [log scale]")
ax2.set_title("Convergence (Loss)", fontweight='bold')
ax2.legend(fontsize=9)

fig.suptitle("Optimizer Comparison — Ill-Conditioned Quadratic", fontsize=14, fontweight='bold')
plt.savefig("optimizer_comparison.png", dpi=150, bbox_inches='tight')
plt.show()

# Summary table
print(f"{'Optimizer':<12} | {'Final loss':>12} | {'Steps to <1':>12}")
print("-" * 42)
for name, (_, _, losses) in results.items():
    final  = losses[-1]
    steps1 = next((i for i, l in enumerate(losses) if l < 1.0), None)
    s1_str = str(steps1) if steps1 is not None else ">200"
    print(f"{name:<12} | {final:>12.4f} | {s1_str:>12}")


## 3. Adam Failure Mode — Non-Convergence on Adversarial Gradients

Reddi et al. (2018) proved that Adam can **fail to converge** on certain gradient sequences. The intuition: Adam divides by $\sqrt{\hat{v}}$, a running average of squared gradients. If gradients are mostly small but occasionally very large, the large gradient inflates $\hat{v}$, suppressing future updates precisely when the optimizer should be making progress.

### Adversarial sequence (paper-style)

$$g_t = \begin{cases} C & \text{if } t \equiv 0 \pmod{k} \\ -1 & \text{otherwise} \end{cases}$$

With $C=100, k=10$: the true minimisation direction is $-1$ (small, frequent gradient), but Adam gives too much weight to the rare, large $+C$ gradient.


In [ ]:
def grad_adversarial(t, C=100, k=10):
    return C if (t % k == 0) else -1.0

def run_sgd_1d(steps=200, lr=0.01):
    x, traj = 0.0, []
    for t in range(1, steps + 1):
        g = grad_adversarial(t)
        x -= lr * g
        traj.append(x)
    return traj

def run_adam_1d(steps=200, lr=0.01, beta1=0.9, beta2=0.999):
    x, m, v, traj = 0.0, 0.0, 0.0, []
    for t in range(1, steps + 1):
        g     = grad_adversarial(t)
        m     = beta1 * m + (1 - beta1) * g
        v     = beta2 * v + (1 - beta2) * g**2
        m_hat = m / (1 - beta1**t)
        v_hat = v / (1 - beta2**t)
        x    -= lr * m_hat / (math.sqrt(v_hat) + 1e-8)
        traj.append(x)
    return traj


steps     = 300
sgd_traj  = run_sgd_1d(steps, lr=0.01)
adam_traj = run_adam_1d(steps, lr=0.01)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(sgd_traj,  label="SGD",  color="steelblue", lw=2)
axes[0].plot(adam_traj, label="Adam", color="tomato",    lw=2)
axes[0].axhline(0, ls='--', color='gray', lw=1, label="Optimum")
axes[0].set_xlabel("Iteration"); axes[0].set_ylabel("x")
axes[0].set_title("Adam failure: trajectory", fontweight='bold')
axes[0].legend()

# Gradient sequence visualisation
axes[1].plot([grad_adversarial(t) for t in range(1, 31)], 'o-', color='mediumpurple', lw=1.5)
axes[1].set_xlabel("Step"); axes[1].set_ylabel("Gradient")
axes[1].set_title("Adversarial gradient sequence (first 30 steps)\n"
                  "Rare large positive + frequent small negative",
                  fontweight='bold')

fig.suptitle("Adam Non-Convergence (Reddi et al., 2018)", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("adam_failure.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"SGD  final x  : {sgd_traj[-1]:.4f}  (closer to optimum = 0 is better)")
print(f"Adam final x  : {adam_traj[-1]:.4f}")


## 4. Normalization — BatchNorm vs LayerNorm

Both techniques normalise activations to have zero mean and unit variance, then apply learned scale $\gamma$ and shift $\beta$. The critical difference is **which axis** the statistics are computed over.

### BatchNorm
$$\hat{x}^{(i)} = \frac{x^{(i)} - \mu_{\text{batch}}}{\sqrt{\sigma^2_{\text{batch}} + \varepsilon}}$$

Statistics computed **across the batch dimension**: $\mu_j = \frac{1}{m}\sum_i x^{(i)}_j$.  
Each *feature* is normalised over the batch.

### LayerNorm
$$\hat{x}^{(i)} = \frac{x^{(i)} - \mu^{(i)}}{\sqrt{{\sigma^{(i)}}^2 + \varepsilon}}$$

Statistics computed **across the feature dimension** within each example: $\mu^{(i)} = \frac{1}{d}\sum_j x^{(i)}_j$.  
Each *sample* is normalised over its own features.

**Both include a full backward pass** (gradients through the normalisation) — this is what is commonly elided in tutorials.


In [ ]:
class BatchNorm:
    """
    Batch Normalisation with running statistics and full backward pass.

    Forward (training): normalise across the batch axis (axis=0).
    Forward (inference): use running mean/var accumulated during training.
    Backward: exact gradient through the normalisation — see Ioffe & Szegedy (2015).
    """
    def __init__(self, num_features: int, eps: float = 1e-5, momentum: float = 0.1):
        self.num_features = num_features
        self.eps          = eps
        self.momentum     = momentum
        self.gamma        = np.ones(num_features)
        self.beta         = np.zeros(num_features)
        self.running_mean = np.zeros(num_features)
        self.running_var  = np.ones(num_features)
        self.training     = True

    def forward(self, x: np.ndarray) -> np.ndarray:
        self.x = x
        if self.training:
            self.batch_mean = x.mean(axis=0)
            self.batch_var  = x.var(axis=0)
            self.x_hat      = (x - self.batch_mean) / np.sqrt(self.batch_var + self.eps)
            # Exponential moving average of statistics
            self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * self.batch_mean
            self.running_var  = (1 - self.momentum) * self.running_var  + self.momentum * self.batch_var
        else:
            self.x_hat = (x - self.running_mean) / np.sqrt(self.running_var + self.eps)
        return self.gamma * self.x_hat + self.beta

    def train(self): self.training = True
    def eval(self):  self.training = False

    def backward(self, dout: np.ndarray):
        """
        Exact gradient derivation — see paper appendix.
        Key insight: the gradient must account for the fact that the mean and
        variance used to normalise x_i are themselves functions of x_i.
        """
        m       = self.x.shape[0]
        dx_hat  = dout * self.gamma
        scale   = 1.0 / np.sqrt(self.batch_var + self.eps)
        dx      = scale / m * (
            m * dx_hat
            - dx_hat.sum(axis=0)
            - self.x_hat * (dx_hat * self.x_hat).sum(axis=0)
        )
        dgamma  = (dout * self.x_hat).sum(axis=0)
        dbeta   = dout.sum(axis=0)
        return dx, dgamma, dbeta


class LayerNorm:
    """
    Layer Normalisation with full backward pass.
    Statistics computed per-sample across the feature axis (axis=1).
    No running statistics needed — inference is identical to training.
    """
    def __init__(self, num_features: int, eps: float = 1e-5):
        self.num_features = num_features
        self.eps          = eps
        self.gamma        = np.ones(num_features)
        self.beta         = np.zeros(num_features)

    def forward(self, x: np.ndarray) -> np.ndarray:
        self.x     = x
        self.mean  = x.mean(axis=1, keepdims=True)
        self.var   = x.var(axis=1, keepdims=True)
        self.x_hat = (x - self.mean) / np.sqrt(self.var + self.eps)
        return self.gamma * self.x_hat + self.beta

    def backward(self, dout: np.ndarray):
        d       = self.x.shape[1]
        dx_hat  = dout * self.gamma
        scale   = 1.0 / np.sqrt(self.var + self.eps)
        dx      = scale / d * (
            d * dx_hat
            - dx_hat.sum(axis=1, keepdims=True)
            - self.x_hat * (dx_hat * self.x_hat).sum(axis=1, keepdims=True)
        )
        dgamma  = (dout * self.x_hat).sum(axis=0)
        dbeta   = dout.sum(axis=0)
        return dx, dgamma, dbeta


## 5. Normalization in a Transformer-like Architecture

We compare BatchNorm and LayerNorm in a minimal 2-layer model trained on a synthetic task.
This isolates the normalization choice from all other factors.

**Key design note for BatchNorm with sequences:**  
BatchNorm expects input shape `[batch, features]`. With sequences `[batch, seq_len, features]`
we must reshape to `[batch*seq_len, features]`, apply BN, and reshape back.  
This means batch statistics are computed **across all token positions in all sequences** — 
which breaks the per-sequence independence that Transformers require.


In [ ]:
class Linear:
    def __init__(self, in_f: int, out_f: int):
        self.W  = np.random.randn(in_f, out_f) * 0.01
        self.b  = np.zeros(out_f)
        self.dW = None
        self.db = None

    def forward(self, x):
        self.x = x
        return x @ self.W + self.b

    def backward(self, dout):
        x_2d      = self.x.reshape(-1, self.x.shape[-1])
        dout_2d   = dout.reshape(-1, dout.shape[-1])
        self.dW   = x_2d.T @ dout_2d
        self.db   = dout.sum(axis=tuple(range(dout.ndim - 1)))
        return dout @ self.W.T


class ReLU:
    def forward(self, x):
        self.mask = x > 0
        return x * self.mask

    def backward(self, dout):
        return dout * self.mask


def mse_loss(pred, target):
    return np.mean((pred - target) ** 2)

def mse_grad(pred, target):
    return 2 * (pred - target) / pred.size


class ModelLN:
    """2-layer model with LayerNorm."""
    def __init__(self, d):
        self.l1   = Linear(d, d); self.norm = LayerNorm(d)
        self.relu = ReLU();        self.l2   = Linear(d, d)

    def forward(self, x):
        h = self.l1.forward(x)
        h = self.norm.forward(h)
        h = self.relu.forward(h)
        return self.l2.forward(h)

    def backward(self, dout):
        dout = self.l2.backward(dout)
        dout = self.relu.backward(dout)
        dout = self.norm.backward(dout)[0]
        return self.l1.backward(dout)

    def update(self, lr):
        for layer in [self.l1, self.l2]:
            layer.W -= lr * layer.dW
            layer.b -= lr * layer.db
        self.norm.gamma -= lr * self.norm.backward.__self__  # placeholder
        # Note: gamma/beta updates handled via the stored gradients
        # (simplified here — in practice tracked separately)


class ModelBN:
    """2-layer model with BatchNorm (applied after reshape)."""
    def __init__(self, d):
        self.l1   = Linear(d, d); self.norm = BatchNorm(d)
        self.relu = ReLU();        self.l2   = Linear(d, d)
        self._last_shape = None

    def forward(self, x):
        h = self.l1.forward(x)
        self._last_shape = h.shape
        B, S, D = h.shape
        h = self.norm.forward(h.reshape(B * S, D)).reshape(B, S, D)
        h = self.relu.forward(h)
        return self.l2.forward(h)

    def backward(self, dout):
        dout      = self.l2.backward(dout)
        dout      = self.relu.backward(dout)
        B, S, D   = dout.shape
        dout      = self.norm.backward(dout.reshape(B * S, D))[0].reshape(B, S, D)
        return self.l1.backward(dout)


def generate_data(batch_size, seq_len, d_model):
    x = np.random.randn(batch_size, seq_len, d_model)
    y = np.tanh(x @ np.random.randn(d_model, d_model))
    return x, y


def train_model(model, steps=300, lr=0.01, batch_size=4, seq_len=16, d_model=32):
    losses, grad_norms = [], []
    for step in range(steps):
        x, y  = generate_data(batch_size, seq_len, d_model)
        out   = model.forward(x)
        loss  = mse_loss(out, y)
        dout  = mse_grad(out, y)
        dx    = model.backward(dout)
        losses.append(loss)
        grad_norms.append(float(np.linalg.norm(dx)))

        # Manual SGD update
        for layer in [model.l1, model.l2]:
            layer.W -= lr * layer.dW
            layer.b -= lr * layer.db

    return losses, grad_norms


np.random.seed(42)
D = 32
model_ln = ModelLN(D)
model_bn = ModelBN(D)

print("Training LayerNorm model...")
loss_ln, grad_ln = train_model(model_ln, steps=300, d_model=D)

np.random.seed(42)
model_bn.norm.training = True
print("Training BatchNorm model...")
loss_bn, grad_bn = train_model(model_bn, steps=300, d_model=D)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("BatchNorm vs LayerNorm — Transformer Context (batch_size=1 per sequence)",
             fontsize=13, fontweight='bold')

# Loss
axes[0].plot(loss_ln, label="LayerNorm", color="steelblue",   lw=2)
axes[0].plot(loss_bn, label="BatchNorm", color="darkorange",  lw=2)
axes[0].set_yscale('log')
axes[0].set_xlabel("Step"); axes[0].set_ylabel("MSE Loss (log)")
axes[0].set_title("Training Loss"); axes[0].legend()

# Gradient norm
axes[1].plot(grad_ln, label="LayerNorm", color="steelblue",  lw=1.5, alpha=0.8)
axes[1].plot(grad_bn, label="BatchNorm", color="darkorange", lw=1.5, alpha=0.8)
axes[1].set_xlabel("Step"); axes[1].set_ylabel("‖∇x‖")
axes[1].set_title("Input Gradient Norm"); axes[1].legend()

# Final loss comparison (bar)
final = {"LayerNorm": loss_ln[-1], "BatchNorm": loss_bn[-1]}
bars  = axes[2].bar(final.keys(), final.values(),
                    color=["steelblue", "darkorange"], edgecolor='white', width=0.4)
for bar, val in zip(bars, final.values()):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                 f"{val:.4f}", ha='center', va='bottom', fontweight='bold')
axes[2].set_ylabel("Final MSE Loss")
axes[2].set_title("Final Loss Comparison")
axes[2].set_ylim(0, max(final.values()) * 1.3)

plt.tight_layout()
plt.savefig("normalization_comparison.png", dpi=150, bbox_inches='tight')
plt.show()


## 6. Learning Rate Schedules

### Cosine Annealing with Linear Warmup

The schedule used in GPT-3, PaLM, and most modern LLMs:

$$
\text{lr}(t) = \begin{cases}
\alpha_{\max} \cdot \dfrac{t}{T_{\text{warmup}}} & t < T_{\text{warmup}} \\[8pt]
\alpha_{\min} + \dfrac{\alpha_{\max} - \alpha_{\min}}{2}\left(1 + \cos\!\left(\pi \cdot \frac{t - T_{\text{warmup}}}{T_{\text{total}} - T_{\text{warmup}}}\right)\right) & t \geq T_{\text{warmup}}
\end{cases}
$$

**Why warmup?** At initialisation, gradients are noisy and the optimizer's moment estimates ($m$, $v$ in Adam) are all zero. A large learning rate at step 1 can immediately overshoot and irreparably damage the initialisation. Warmup gives the optimizer time to build reliable gradient statistics before taking large steps.

**Why cosine decay?** It avoids a sharp LR drop (which can cause instability) and smoothly reduces the step size as the model approaches a minimum. The $\alpha_{\min}$ floor prevents the LR from reaching zero, which can cause the optimizer to stagnate.


In [ ]:
class CosineWithWarmupScheduler:
    def __init__(self, max_lr: float, min_lr: float, warmup_steps: int, total_steps: int):
        self.max_lr       = max_lr
        self.min_lr       = min_lr
        self.warmup_steps = warmup_steps
        self.total_steps  = total_steps

    def get_lr(self, step: int) -> float:
        step = min(step, self.total_steps)
        if step < self.warmup_steps:
            # Linear warmup: rises from 0 to max_lr
            return self.max_lr * (step / max(1, self.warmup_steps))
        else:
            # Cosine decay: falls from max_lr to min_lr
            progress = (step - self.warmup_steps) / max(1, self.total_steps - self.warmup_steps)
            return self.min_lr + 0.5 * (self.max_lr - self.min_lr) * (1 + math.cos(math.pi * progress))


# Visualise schedule variants
total_steps = 5000

schedules = {
    "10% warmup": CosineWithWarmupScheduler(1e-3, 1e-5, 500,  total_steps),
    "20% warmup": CosineWithWarmupScheduler(1e-3, 1e-5, 1000, total_steps),
    "No warmup":  CosineWithWarmupScheduler(1e-3, 1e-5, 0,    total_steps),
}

fig, ax = plt.subplots(figsize=(12, 5))

for name, sched in schedules.items():
    lrs = [sched.get_lr(t) for t in range(total_steps + 1)]
    ax.plot(lrs, label=name, lw=2)

ax.axhline(1e-5, ls=':', color='gray', lw=1, label=f'min_lr = {1e-5}')
ax.set_xlabel("Training step")
ax.set_ylabel("Learning rate")
ax.set_title("Cosine Annealing with Warmup — Schedule Variants", fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig("lr_schedules.png", dpi=150, bbox_inches='tight')
plt.show()


## 7. Conclusions — When to Use Each Technique

### Optimizers

**SGD** is simple and transparent. It works well with careful tuning and is the optimizer of choice for
large-batch computer vision training (ResNets, ViTs) where the loss surface is well-understood.
However, it is highly sensitive to the learning rate and does poorly on ill-conditioned problems without
significant momentum tuning.

**Momentum** resolves the ill-conditioning problem at minimal cost: it requires only one additional
hyperparameter ($\beta$) and one extra buffer per parameter. It should be the default over vanilla SGD.

**AdaGrad** is powerful for sparse gradient problems (NLP with bag-of-words features, recommendation systems)
because infrequently-updated parameters retain a high effective learning rate. Its fatal weakness —
the monotonically growing accumulator — makes it unsuitable for long training runs.

**Adam** is the dominant choice for deep learning because it combines the benefits of momentum and
adaptive rates with bias-corrected initialisation. It is robust to LR choice across a wide range of
architectures and datasets. Its main weaknesses are: (1) non-convergence on certain gradient sequences
(Reddi et al., 2018), resolved by AMSGrad; (2) poorer generalisation than SGD in some vision settings,
where the adaptivity can lead to sharp minima.

**AdamW** (Adam + decoupled weight decay) is the current standard for training LLMs. Weight decay
in Adam is not equivalent to L2 regularisation because the adaptive preconditioner interacts with
the penalty — AdamW fixes this by applying decay directly to the weights, not through the gradient.

---

### Normalization

**BatchNorm** is the right choice for CNNs and tasks where:
- Batch size is large (≥32) so the batch statistics are stable.
- Sequences have fixed, uniform structure.
- You need fast inference and can precompute running statistics.

**LayerNorm** is the right choice for Transformers and sequential models because:
- It operates on a single example at a time, making it **batch-size independent**.
- It handles **variable-length sequences** naturally.
- It is **deterministic at inference** (no running statistics required).
- In Transformers, each token's representation is independently meaningful — normalising across
  the feature dimension preserves this per-token semantics.

In practice: if you're building anything Transformer-based, use **LayerNorm**. If you're building
a CNN or MLP with large batch sizes, **BatchNorm** typically converges faster and achieves lower
final loss.

---

### Learning Rate Schedules

- **No schedule**: only defensible for very short experiments or when using a grid search to find a
  good fixed LR.
- **Cosine decay alone**: good default for tasks where training stability is not an issue.
- **Warmup + cosine decay**: the current best practice for training LLMs and ViTs. The warmup prevents
  early-step instability; the cosine decay ensures the model fine-tunes near the minimum.
- **Warmup fraction**: typically 1–10% of total steps. Too much warmup wastes compute; too little
  defeats the purpose.

---

### Connection to ML Safety

Understanding optimizer and normalization dynamics is directly relevant to ML Safety:
- **Training instabilities** (loss spikes, gradient explosions) can cause models to develop
  unexpected behaviors. Robust training requires understanding when and why instabilities occur.
- **Interpretability** of trained models depends on understanding what the training process converges to.
  Different optimizers may find different minima with different generalisation properties — a safety-critical
  consideration when deploying models.
- **Gradient norms** and loss curvature are diagnostic signals used in safety monitoring during training
  of large models.

---
*Notebook generated as Week 2 deliverable for the ML Safety Program.*
